# Adaptation Under Perturbation -- trim-development case study

The standard transit-design benchmark (§12 of `evaluation_seeded_lc_bco_mumford0`)
evaluates each method from a fresh NX-heuristic initialisation. That setting
favours wholesale-rebuild operators (e.g. neural BCO type-1 reconstructs
routes from scratch). It tells us little about the situation real operators
face: **adapt an existing, already-deployed network to a perturbed instance**
(demand shift, edge unavailability) **while disturbing it as little as
possible**.

This notebook is that experiment. We:

1. Fix a stable network `R_0` (NX-heuristic on the unperturbed Mumford0).
2. Apply three perturbations -- `control` (no change), `demand` (cluster
   demand spike), `edge` (two random edges removed).
3. For each perturbation, run every method **starting from `R_0`** on the
   **perturbed tensors**, and record:
   - **`cost`** -- the scalar objective;
   - **`mean adjustment_degree`** -- the per-route `1 - alignment-similarity`
     from `R_0`, computed by
     `connectpt.routes_generator.bee_colony.get_adjustment_degrees(...,
     mode="current")`. **Measured post-hoc only; it does NOT enter the
     optimization objective** (`adjustment_degree_weight` stays 0).

Results land in `artifacts/results/adaptation_*` so they do not touch the
files produced by the main eval notebook.

Hypothesis: trim-based methods (the RL edit policy and trim-BCO variants)
achieve a comparable cost with a much smaller `mean adjustment_degree` --
i.e. they adapt with fewer route-set changes than wholesale-rebuild
neural BCO. The headline figure is the cost-vs-adjustment Pareto scatter.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

# eval_lib must be importable -- the notebook's working directory holds it.
import os as _os, sys as _sys
_NB_DIR = _os.path.abspath(".")
if _NB_DIR not in _sys.path:
    _sys.path.insert(0, _NB_DIR)
import eval_lib
from eval_lib import *
from eval_lib import plots as _plots
from eval_lib import _run_baseline  # underscore name, skipped by `import *`

# Library bits not re-exported by eval_lib.
from connectpt.routes_generator import CityGraphData, build_nx_heuristic_routes
from connectpt.routes_generator.bee_colony import get_adjustment_degrees

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print("eval_lib OK; RESULTS_DIR =", RESULTS_DIR.relative_to(ROOT_DIR))


## 1. Base instance and stable network `R_0`

`R_0` is the NX-heuristic initial route set on the *unperturbed* Mumford0
instance (deterministic, seed 0). This is the network the operator is
*already running* before the perturbation hits -- every method below starts
from it.

Eval bounds match the benchmark sweep for Mumford0 (`n_routes=12,
min_route_len=2, max_route_len=15`).

In [ ]:
CITY = "Mumford0"
ADAPT_N_ROUTES = 12
ADAPT_MIN_LEN = 2
ADAPT_MAX_LEN = 15

BASE_TENSORS = load_benchmark_tensors(CITY)
print({k: tuple(v.shape) for k, v in BASE_TENSORS.items()})

_base_graph = CityGraphData.from_tensors(
    BASE_TENSORS["node_locs"], BASE_TENSORS["street_adj"], BASE_TENSORS["demand"],
    pos_only=False,
)
R_0 = build_nx_heuristic_routes(
    _base_graph,
    num_routes=ADAPT_N_ROUTES,
    min_len=ADAPT_MIN_LEN,
    max_len=ADAPT_MAX_LEN,
    seed=0,
)
print(f"R_0 shape: {tuple(R_0.shape)}")


## 2. Perturbations

Three instances of the Mumford0 problem:

* **`control`** -- identical to the base tensors. Sanity-checks every method
  to confirm `adjustment_degree ~ 0` when nothing changed.
* **`demand`** -- pick a geometric cluster of 3 nearby nodes and multiply the
  demand to/from them by 1.5x. Models a sudden hotspot.
* **`edge`** -- remove 2 random `street_adj` edges that are *currently used*
  by at least one route of `R_0`. Models a road closure.

The functions return cloned tensor dicts so the base instance is never
mutated.

In [ ]:
def perturb_control(tensors, seed=0):
    """No change -- calibration baseline (every metric should mirror initial)."""
    return ({k: v.clone() if torch.is_tensor(v) else v
             for k, v in tensors.items()}, {})


def perturb_demand(tensors, seed, factor=1.5, cluster_size=3):
    """Multiply demand by `factor` for a small cluster of geometrically nearby
    nodes around a random anchor."""
    rng = np.random.default_rng(seed)
    locs = tensors["node_locs"].cpu().numpy()
    anchor = int(rng.integers(0, locs.shape[0]))
    dists = np.linalg.norm(locs - locs[anchor], axis=1)
    cluster = np.argsort(dists)[:cluster_size].tolist()
    new = {k: v.clone() if torch.is_tensor(v) else v for k, v in tensors.items()}
    demand = new["demand"]
    demand[cluster, :] *= factor
    demand[:, cluster] *= factor
    new["demand"] = demand
    return new, {"anchor": anchor, "cluster": cluster, "factor": factor}


def perturb_edge(tensors, seed, n_remove=2, routes=None):
    """Remove `n_remove` undirected edges from street_adj. If `routes` is
    given, prefer edges actually used by those routes."""
    rng = np.random.default_rng(seed)
    new = {k: v.clone() if torch.is_tensor(v) else v for k, v in tensors.items()}
    sa = new["street_adj"]
    n = sa.shape[0]
    existing = [(i, j) for i in range(n) for j in range(i + 1, n)
                if float(sa[i, j]) > 0]
    if routes is not None:
        used = set()
        r_flat = routes.view(-1, routes.shape[-1])
        for r in r_flat:
            seq = [int(x) for x in r.tolist() if x >= 0]
            for a, b in zip(seq[:-1], seq[1:]):
                used.add((min(a, b), max(a, b)))
        used_existing = [e for e in existing if e in used]
        if len(used_existing) >= n_remove:
            existing = used_existing
    pick = rng.choice(len(existing), size=min(n_remove, len(existing)),
                      replace=False)
    removed = [existing[int(i)] for i in pick]
    for i, j in removed:
        sa[i, j] = 0
        sa[j, i] = 0
    new["street_adj"] = sa
    return new, {"removed_edges": removed}


PERTURBATIONS = {}
PERTURBATION_INFO = {}
PERTURBATIONS["control"], PERTURBATION_INFO["control"] = perturb_control(BASE_TENSORS)
PERTURBATIONS["demand"], PERTURBATION_INFO["demand"] = perturb_demand(
    BASE_TENSORS, seed=42, factor=1.5, cluster_size=3)
PERTURBATIONS["edge"], PERTURBATION_INFO["edge"] = perturb_edge(
    BASE_TENSORS, seed=42, n_remove=2, routes=R_0)

for name, info in PERTURBATION_INFO.items():
    print(f"  {name:8s}: {info}")


## 3. Methods

We compare:

* `Initial network` -- evaluate `R_0` as-is on the perturbed tensors (the
  "do nothing" baseline; its `adjustment_degree == 0` by construction).
* `RL improvement only` -- the trained edit policy applied greedily to `R_0`.
* All six BCO variants from `BCO_VARIANTS` (BCO, neural BCO, the four
  trim/edit/construction blends), each in `without_worse` mode.

`worse` mode is skipped to keep the runtime manageable (6 BCO variants x 3
perturbations is already a substantial sweep).

In [ ]:
ADAPT_METHOD_SPECS = (
    [INITIAL_METHOD]
    + ([RL_ONLY_METHOD] if RUN_RL_ONLY_BASELINE else [])
    + [bco_method(v) for v in BCO_VARIANTS]
)
print("Methods to compare:")
for m in ADAPT_METHOD_SPECS:
    print(f"  - {m.get('label', m['kind'])} ({m['kind']})")


## 4. Run adaptation

Each (perturbation x method) call goes through the unified `run_method` and
produces a uniform `RunResult`. Initial routes are always `R_0`; the
*perturbed* tensors are passed in via `tensors=`. `run_name_scope` keeps
the cfg labels distinct per perturbation.

Per-method run-times: BCO with the current `n_iterations=150` on Mumford0
takes seconds; 7 methods x 3 perturbations is roughly a few minutes.

In [ ]:
ADAPT_ACCEPT_MODE = "without_worse"
ADAPT_RESULTS = {}

for pname, ptensors in PERTURBATIONS.items():
    print(f"\n=== Perturbation: {pname} ===")
    ADAPT_RESULTS[pname] = []
    for spec in ADAPT_METHOD_SPECS:
        label = spec.get("label", spec["kind"])
        print(f"  method: {label}")
        try:
            result = run_method(
                spec,
                init_routes=R_0,
                tensors=ptensors,
                n_routes=ADAPT_N_ROUTES,
                min_route_len=ADAPT_MIN_LEN,
                max_route_len=ADAPT_MAX_LEN,
                accept_mode=ADAPT_ACCEPT_MODE,
                seed=0,
                dataset=f"{CITY} {pname}",
                run_name_scope=f"adaptation_{pname}_",
            )
        except Exception as exc:
            print(f"    FAILED: {exc!r}")
            continue
        ADAPT_RESULTS[pname].append(result)


## 5. Post-hoc adjustment degree

For every `RunResult` we compute the **mean per-route adjustment degree**
relative to `R_0`:

```
deg = get_adjustment_degrees(R_final, R_0, symmetric_routes=True, mode="current")
mean_adjustment_degree = deg.mean()
```

It lives in `[0, 1]`: `0` = the route-set is unchanged from `R_0`, `1` =
every route is fully different (no aligned subsequence). This is the
**already-in-the-codebase** metric (`bee_colony.py:150-168`). The penalty
hook (`adjustment_degree_weight`) is left at 0 -- the metric only reports,
it does not steer the optimizer.

The aggregated table is saved to `artifacts/results/adaptation_pareto.csv`.

In [ ]:
SYMMETRIC_ROUTES = True  # mirrors experiment/standard.yaml


def _pad_routes_to(routes, n_routes, max_route_len):
    """Pad a ``[1, R, L]`` route tensor with ``-1`` along axes 1/2 so it has
    shape ``[1, n_routes, max_route_len]``. ``get_adjustment_degrees`` requires
    candidate and reference to share an identical shape -- each tensor is only
    padded up to its own longest route, so a candidate that produced a 13-stop
    longest route and a reference with a 15-stop longest route mismatch unless
    we pad them up to a common envelope before comparing."""
    cur_routes, cur_len = routes.shape[-2], routes.shape[-1]
    pad_routes = max(0, n_routes - cur_routes)
    pad_len = max(0, max_route_len - cur_len)
    if pad_routes == 0 and pad_len == 0:
        return routes
    # F.pad pads from the last dim outward: (last_left, last_right, prev_left,
    # prev_right, ...). We pad the trailing slots on both route-axis and
    # length-axis.
    return torch.nn.functional.pad(
        routes, (0, pad_len, 0, pad_routes), value=-1)


def mean_adjustment_degree(routes, reference):
    """Mean per-route adjustment degree vs ``reference``, mode ``current``.

    Both tensors are padded with ``-1`` up to a common ``[1, R, L]`` envelope
    so the alignment-score shape check in ``bee_colony._get_alignment_scores``
    passes when the two route sets have different longest-route lengths.
    """
    cand = as_route_tensor(routes)
    ref = as_route_tensor(reference)
    if cand.ndim == 2:
        cand = cand.unsqueeze(0)
    if ref.ndim == 2:
        ref = ref.unsqueeze(0)
    n_routes = max(cand.shape[-2], ref.shape[-2])
    max_route_len = max(cand.shape[-1], ref.shape[-1])
    cand = _pad_routes_to(cand, n_routes, max_route_len)
    ref = _pad_routes_to(ref, n_routes, max_route_len)
    degrees = get_adjustment_degrees(cand, ref, SYMMETRIC_ROUTES, mode="current")
    return float(degrees.mean().item())


_rows = []
for pname, results in ADAPT_RESULTS.items():
    for r in results:
        _rows.append({
            "perturbation": pname,
            "method": r.label,
            "kind": r.kind,
            "cost": metric_value(r.metrics, "cost"),
            "ATT": metric_value(r.metrics, "ATT"),
            "RTT": metric_value(r.metrics, "RTT"),
            "d_un": metric_value(r.metrics, "$d_{un}$"),
            "mean_adjustment_degree": mean_adjustment_degree(r.routes, R_0),
        })

adaptation_df = pd.DataFrame(_rows)
save_table(adaptation_df, "adaptation_pareto")
display(adaptation_df.round(4))

## 6. Pareto scatter: cost vs adjustment degree

For each perturbation, each method lands as one point: x = mean adjustment
degree from `R_0`, y = cost. The hypothesis is that trim-based methods
(`Extend/trim edit-only`, `Heuristic + extend/trim split`, and especially
`RL improvement only`) sit further left (smaller change from `R_0`) than
the wholesale-rebuild `neural BCO`, at comparable cost.

The figure is built from the CSV above -- no separate route-data dump
needed for this view.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5), constrained_layout=True)
markers = {"control": "o", "demand": "s", "edge": "D"}
methods = sorted(adaptation_df["method"].unique())
cmap = plt.get_cmap("tab10")
mcolors = {m: cmap(i % 10) for i, m in enumerate(methods)}

for _, row in adaptation_df.iterrows():
    ax.scatter(row["mean_adjustment_degree"], row["cost"],
               marker=markers.get(row["perturbation"], "x"),
               color=mcolors[row["method"]],
               s=90, edgecolor="black", linewidth=0.6)

# legend: method colour + perturbation marker, de-duplicated
from matplotlib.lines import Line2D
method_handles = [Line2D([0], [0], marker="o", color="w",
                          markerfacecolor=mcolors[m], markersize=9,
                          markeredgecolor="black", label=m)
                  for m in methods]
pert_handles = [Line2D([0], [0], marker=mk, color="dimgray", lw=0,
                        markersize=9, label=pn)
                for pn, mk in markers.items()]
leg1 = ax.legend(handles=method_handles, fontsize=8, loc="upper left",
                 title="method", framealpha=0.9)
ax.add_artist(leg1)
ax.legend(handles=pert_handles, fontsize=9, loc="lower right",
          title="perturbation", framealpha=0.9)

ax.set_xlabel("mean adjustment degree vs R_0  (0 = unchanged, 1 = fully changed)")
ax.set_ylabel("cost  (lower = better)")
ax.set_title(f"{CITY} adaptation: cost vs change-from-reference\n"
             f"(adjustment_degree measured post-hoc, NOT in objective)")
ax.grid(alpha=0.25)
ax.set_xlim(left=-0.02)
plt.show()


## 7. Route-comparison figures

One figure per perturbation: cell 0 = `Initial network` (= `R_0` evaluated on
the perturbed instance), cells 1.. = each method's adapted routes drawn as
diffs vs `R_0`. The shared `render_route_comparison_figure` is used; route
**data** (not images) is saved to `artifacts/results/adaptation_<pert>_routes.pt`
through the `save_as=` hook -- regenerate the figure later with
`redraw_route_results("adaptation_<pert>")`.

Note: the `edge`-perturbation figure draws the *perturbed* street graph
(removed edges are absent from the underlying lines), which makes the
diffs over removed corridors easy to spot.

In [ ]:
for pname, results in ADAPT_RESULTS.items():
    if not results:
        continue
    reference_run = next((r for r in results if r.kind == "initial"), results[0])
    cases = [r for r in results if r is not reference_run]
    render_route_comparison_figure(
        reference_run,
        cases,
        PERTURBATIONS[pname]["node_locs"],
        PERTURBATIONS[pname]["street_adj"],
        title=f"{CITY} adaptation -- perturbation: {pname}",
        ncols=min(4, 1 + len(cases)),
        palette="tab10",
        with_overlap_curves=False,
        save_as=f"adaptation_{pname}",
    )
    plt.show()


## What you should see

* `control`: every method should produce small `adjustment_degree` and a
  cost close to whatever it achieves on the unperturbed Mumford0. This
  calibrates the metric.
* `demand`: methods that can *add coverage* to the new hotspot should
  reduce d_un with small adjustment. `Extend/trim edit-only` and
  `Heuristic + extend/trim split` are the natural candidates.
* `edge`: removing an actively-used edge forces detours. Trim-aware methods
  should shorten the affected route(s) instead of rebuilding everything.

If the Pareto scatter splits the methods into two clusters
(low-adjustment / moderate-cost trim cluster vs high-adjustment /
low-cost neural-BCO cluster), the "operator-friendly" framing of the
trim development is empirically established on this instance.